# RealWaste classification: SimpleCNN, MultiScaleCNN and EfficientNet-B0

Nine-class waste classification on RealWaste (4,752 photos). This notebook holds all the code of the project:
data split, data pipeline, the three models, the training loop and the final test evaluation.

The models were trained earlier; the training logs are the development notebooks `05` to `10` in `notebooks/`. Running this notebook again does not retrain
them: the training function finds the saved checkpoints and stops, and the test evaluation is recomputed from the saved
weights and compared with the stored results. Runtime: T4 GPU, about 15 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import hashlib, json, random, shutil, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_recall_fscore_support
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

ROOT = Path("/content/drive/MyDrive/Deep Learning - RealWaste")
DATA_DIR = ROOT / "01_Dataset/RealWaste"
SPLIT_FILE = ROOT / "02_Splits/split_grouped_v2.csv"
SPLIT_SHA256 = "65dc85e83acee56f5db0a0f3b9b5f08c84cf71d3d35c76f69c5213d4a33b3d3d"
CKPT_DIR = ROOT / "03_Checkpoints"
RESULT_DIR = ROOT / "04_Results/experiments"

SEEDS = [42, 43, 44]
BATCH_SIZE = 32
PATIENCE = 5
EPOCHS = {"E1": 40, "E2": 40, "E4": 40, "E3_stageA": 10, "E3_stageB": 15}
NUM_WORKERS = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = device.type == "cuda"

## 1. Dataset

Every class folder is counted. The dataset has 4,752 RGB JPEG images of 524 x 524 pixels in 9 classes.

In [ ]:
counts = {d.name: len(list(d.glob("*.jpg"))) for d in sorted(DATA_DIR.iterdir()) if d.is_dir()}
print(counts, "total:", sum(counts.values()))
pd.Series(counts).sort_values().plot.barh(title="Images per class")
plt.show()

## 2. Grouped train / validation / test split

The same object is often photographed 2 or 3 times under neighbouring file numbers (for example `Plastic_644` and
`Plastic_646`). A random split would put one photo in training and the other in the test set. So each class is cut
into blocks of 10 consecutive file numbers, and whole blocks are assigned 70 / 15 / 15 with seed 42.

The split was saved once. Here it is generated again and compared with the saved file, whose SHA-256 hash is checked.

In [ ]:
def grouped_split(df, block_size=10, val_ratio=0.15, test_ratio=0.15, seed=42):
    df = df.sort_values("filepath").reset_index(drop=True)
    number = df["filepath"].str.extract(r"_(\d+)\.[A-Za-z]+$", expand=False).astype(int)
    df["block"] = number // block_size
    rng = np.random.default_rng(seed)
    split_of = {}
    for cls in sorted(df["class_name"].unique()):
        blocks = rng.permutation(np.sort(df.loc[df["class_name"] == cls, "block"].unique()))
        n_test, n_val = round(test_ratio * len(blocks)), round(val_ratio * len(blocks))
        for i, b in enumerate(blocks):
            split_of[cls, b] = "test" if i < n_test else "val" if i < n_test + n_val else "train"
    df["split"] = [split_of[key] for key in zip(df["class_name"], df["block"])]
    return df


assert hashlib.sha256(SPLIT_FILE.read_bytes()).hexdigest() == SPLIT_SHA256, "the split file has changed"
split_df = pd.read_csv(SPLIT_FILE)
again = grouped_split(split_df[["filepath", "class_name"]]).set_index("filepath")["split"]
print("generated split equals the saved one:", again.eq(split_df.set_index("filepath")["split"]).all())
print(split_df["split"].value_counts())

### Duplicate check (summary)

To find near-duplicates we compared a difference hash (dHash) of every image: the image is shrunk to 17 x 16 grey
pixels and each pixel is compared with its right neighbour, giving 256 bits. Two photos of the same object give few
differing bits. The full check is in `notebooks/03_duplicate_check.ipynb`; two pairs found there:

In [ ]:
def dhash(path, size=16):
    pixels = np.asarray(Image.open(path).convert("L").resize((size + 1, size), Image.Resampling.LANCZOS), dtype=np.int16)
    return pixels[:, 1:] > pixels[:, :-1]


for a, b in [("Metal/Metal_216.jpg", "Metal/Metal_217.jpg"), ("Cardboard/Cardboard_140.jpg", "Cardboard/Cardboard_142.jpg")]:
    print(a, b, "differing bits:", (dhash(DATA_DIR / a) != dhash(DATA_DIR / b)).sum(), "of 256")

## 3. Data pipeline

Training images get random crop, flip, rotation and colour changes; validation and test images are only resized.
All images are normalised with the ImageNet mean and standard deviation, which EfficientNet-B0 expects.

In [ ]:
mapping = pd.read_csv(ROOT / "02_Splits/class_mapping.csv")
classes = mapping.sort_values("class_index")["class_name"].tolist()
split_df["label"] = split_df["class_name"].map(dict(zip(mapping["class_name"], mapping["class_index"])))
split_df["image_path"] = [str(DATA_DIR / c / Path(f).name) for c, f in zip(split_df["class_name"], split_df["filepath"])]

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.0),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])


class WasteDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = Image.open(row.image_path).convert("RGB")
        return self.transform(image), row.label, row.filepath


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    # every loader worker gets its own random state, derived from the loader's seed
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


def make_loader(subset, transform, seed, shuffle=False):
    data = WasteDataset(split_df[split_df["split"] == subset], transform)
    return DataLoader(data, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      persistent_workers=NUM_WORKERS > 0, pin_memory=device.type == "cuda",
                      worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(seed))

## 4. Models

- **E1 SimpleCNN**: 3 blocks of conv 3x3, batch norm, ReLU and max pooling, then global average pooling and a small
  dense head (111,145 parameters).
- **E2 MultiScaleCNN**: each block runs 1x1, 3x3 and 5x5 convolutions side by side and joins them (1,230,377 parameters).
- **E3 EfficientNet-B0** pretrained on ImageNet with a new 9-class head (4,019,077 parameters).
- **E4** is E2 trained without augmentation.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=9, dropout=0.3, widths=(32, 64, 128)):
        super().__init__()
        layers, c_in = [], 3
        for c_out in widths:
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out),
                       nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(c_in, 128), nn.ReLU(inplace=True),
                                        nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


class MultiScaleBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        c1 = c5 = c_out // 4
        self.branch1 = nn.Conv2d(c_in, c1, 1, bias=False)
        self.branch3 = nn.Conv2d(c_in, c_out - c1 - c5, 3, padding=1, bias=False)
        self.branch5 = nn.Conv2d(c_in, c5, 5, padding=2, bias=False)
        self.bn = nn.BatchNorm2d(c_out)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(torch.cat([self.branch1(x), self.branch3(x), self.branch5(x)], dim=1)))


class MultiScaleCNN(nn.Module):
    def __init__(self, num_classes=9, dropout=0.3, stem_width=32, widths=(64, 128, 256, 256)):
        super().__init__()
        layers = [nn.Conv2d(3, stem_width, 3, padding=1, bias=False), nn.BatchNorm2d(stem_width),
                  nn.ReLU(inplace=True), nn.MaxPool2d(2)]
        c_in = stem_width
        for c_out in widths:
            layers += [MultiScaleBlock(c_in, c_out), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(c_in, 128), nn.ReLU(inplace=True),
                                        nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


class EfficientNetB0Transfer(nn.Module):
    def __init__(self, num_classes=9, dropout=0.2, pretrained=False):
        super().__init__()
        self.net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        self.net.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(1280, num_classes))
        self.trainable_blocks = len(self.net.features)

    def set_trainable_blocks(self, n):
        # freeze every feature block except the last n; the head always trains
        self.trainable_blocks = n
        for i, block in enumerate(self.net.features):
            block.requires_grad_(i >= len(self.net.features) - n)
        self.train(self.training)

    def train(self, mode=True):
        # frozen blocks stay in eval mode, so their batch norm keeps the ImageNet statistics
        super().train(mode)
        for block in self.net.features[:len(self.net.features) - self.trainable_blocks]:
            block.eval()
        return self

    def forward(self, x):
        return self.net(x)


MODELS = {"E1": SimpleCNN, "E2": MultiScaleCNN, "E3": EfficientNetB0Transfer, "E4": MultiScaleCNN}
for name, model_class in MODELS.items():
    print(name, f"{sum(p.numel() for p in model_class().parameters()):,} parameters")

## 5. Training

All models use cross-entropy, AdamW (learning rate 1e-3, weight decay 1e-4), batch size 32 and mixed precision.
The learning rate is halved when validation macro-F1 has not improved for 2 epochs, and training stops after 5 epochs
without improvement. The epoch with the best validation macro-F1 is kept.

After every epoch the full training state is saved, so a Colab disconnect only costs one epoch. A run that already
finished is skipped, which is what happens when this notebook is run again.

In [ ]:
def macro_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(len(classes)),
                                                  average="macro", zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)), "macro_precision": float(p),
            "macro_recall": float(r), "macro_f1": float(f1)}


@torch.no_grad()
def predict(model, loader, criterion=None):
    model.eval()
    probs, labels, paths, loss = [], [], [], 0.0
    for x, y, p in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device.type, enabled=AMP):
            logits = model(x).float()
        if criterion is not None:
            loss += criterion(logits, y).item() * len(y)
        probs.append(torch.softmax(logits, dim=1).cpu())
        labels.append(y.cpu())
        paths += p
    probs, y_true = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    return {"y_true": y_true, "y_pred": probs.argmax(1), "confidence": probs.max(1), "paths": paths,
            "loss": loss / len(y_true)}


def fit(model, train_loader, val_loader, lr, max_epochs, ckpt_dir, name, history_path):
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_path, last_path = ckpt_dir / f"{name}_best.pt", ckpt_dir / f"{name}_last.pt"
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    scaler = torch.amp.GradScaler(device.type, enabled=AMP)
    history, best_f1, best_epoch, stale, start = [], -1.0, 0, 0, 1

    if last_path.exists():  # continue an interrupted run, or skip a finished one
        state = torch.load(last_path, map_location=device)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        history, best_f1, best_epoch = state["history"], state["best_val_macro_f1"], state["best_epoch"]
        stale, start = state["epochs_without_improvement"], state["epoch"] + 1
        if stale >= PATIENCE or state["epoch"] >= max_epochs:
            print(f"{name}: already finished at epoch {state['epoch']}, best epoch {best_epoch}")
            return pd.DataFrame(history)

    for epoch in range(start, max_epochs + 1):
        t0 = time.time()
        model.train()
        loss_sum, correct = 0.0, 0
        for x, y, _ in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            with torch.autocast(device.type, enabled=AMP):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()

        n = len(train_loader.dataset)
        val = predict(model, val_loader, criterion)
        m = macro_metrics(val["y_true"], val["y_pred"])
        history.append({"epoch": epoch, "train_loss": round(loss_sum / n, 5), "train_accuracy": round(correct / n, 5),
                        "val_loss": round(val["loss"], 5), "val_accuracy": round(m["accuracy"], 5),
                        "val_macro_f1": round(m["macro_f1"], 5), "lr": optimizer.param_groups[0]["lr"],
                        "seconds": round(time.time() - t0, 1)})
        if history[-1]["val_macro_f1"] > best_f1:
            best_f1, best_epoch, stale = history[-1]["val_macro_f1"], epoch, 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_metrics": m}, best_path)
        else:
            stale += 1
        scheduler.step(history[-1]["val_macro_f1"])

        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch,
                    "history": history, "best_val_macro_f1": best_f1, "best_epoch": best_epoch,
                    "epochs_without_improvement": stale}, last_path)
        history_path.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(f"{name} epoch {epoch}: train loss {loss_sum / n:.4f}, val loss {val['loss']:.4f}, "
              f"val macro-F1 {m['macro_f1']:.4f}")
        if stale >= PATIENCE:
            break
    return pd.DataFrame(history)

In [ ]:
def run_dirs(exp, seed):
    # seed 42 is the original run; the other seeds live in trials/seedNN
    sub = "" if seed == 42 else f"trials/seed{seed}"
    return CKPT_DIR / exp / sub, RESULT_DIR / exp / sub


def train_run(exp, seed):
    ckpt, res = run_dirs(exp, seed)
    train_loader = make_loader("train", eval_tf if exp == "E4" else train_tf, seed, shuffle=True)
    val_loader = make_loader("val", eval_tf, seed)
    seed_everything(seed)

    if exp != "E3":
        model = MODELS[exp]().to(device)
        fit(model, train_loader, val_loader, 1e-3, EPOCHS[exp], ckpt, exp, res / f"{exp}_history.csv")
        return

    # E3 stage A: frozen backbone, only the new head learns
    model = EfficientNetB0Transfer(pretrained=True).to(device)
    model.set_trainable_blocks(0)
    a = fit(model, train_loader, val_loader, 1e-3, EPOCHS["E3_stageA"], ckpt, "E3_stageA", res / "E3_stageA_history.csv")
    # E3 stage B: start from the best stage-A weights and fine-tune the last 3 blocks with a smaller learning rate
    model.load_state_dict(torch.load(ckpt / "E3_stageA_best.pt", map_location=device)["model"])
    model.set_trainable_blocks(3)
    b = fit(model, train_loader, val_loader, 1e-4, EPOCHS["E3_stageB"], ckpt, "E3_stageB", res / "E3_stageB_history.csv")
    best_stage = "B" if b["val_macro_f1"].max() > a["val_macro_f1"].max() else "A"
    if not (ckpt / "E3_best.pt").exists():
        shutil.copyfile(ckpt / f"E3_stage{best_stage}_best.pt", ckpt / "E3_best.pt")


for exp in ["E1", "E2", "E3", "E4"]:
    for seed in SEEDS if exp != "E4" else [42]:
        train_run(exp, seed)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, exp in zip(axes, ["E1", "E2", "E3"]):
    h = pd.read_csv(RESULT_DIR / exp / f"{exp}_history.csv")
    ax.plot(h["epoch"], h["train_loss"], label="train loss")
    ax.plot(h["epoch"], h["val_loss"], label="validation loss")
    ax.plot(h["epoch"], h["val_macro_f1"], label="validation macro-F1")
    ax.set(title=f"{exp}, seed 42", xlabel="epoch")
    ax.legend()
plt.show()

## 6. Test evaluation

Each saved model is evaluated on the 707 test images. The last column compares the macro-F1 with the value stored by
the final evaluation run (`09_final_evaluation`), to show that this code gives the same numbers.

In [ ]:
test_loader = make_loader("test", eval_tf, 42)
rows, outputs = [], {}
for exp in ["E1", "E2", "E3", "E4"]:
    for seed in SEEDS if exp != "E4" else [42]:
        ckpt, res = run_dirs(exp, seed)
        state = torch.load(ckpt / f"{exp}_best.pt", map_location=device)
        model = MODELS[exp]().to(device)
        model.load_state_dict(state["model"])
        out = predict(model, test_loader)
        stored = res / f"{exp}_metrics.json" if seed == 42 else RESULT_DIR / exp / f"test_seeds/seed{seed}/{exp}_metrics.json"
        rows.append({"model": exp, "seed": seed, "val_macro_f1": state["val_metrics"]["macro_f1"],
                     **macro_metrics(out["y_true"], out["y_pred"]),
                     "stored_macro_f1": json.loads(stored.read_text())["macro_f1"] if stored.exists() else None})
        if seed == 42:
            outputs[exp] = out

results = pd.DataFrame(rows)
print(results.round(4).to_string(index=False))
metric_cols = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "val_macro_f1"]
results.groupby("model")[metric_cols].agg(["mean", "std"]).round(4)

In [ ]:
f1_per_class = {exp: precision_recall_fscore_support(o["y_true"], o["y_pred"], labels=range(len(classes)),
                                                     zero_division=0)[2] for exp, o in outputs.items()}
print(pd.DataFrame(f1_per_class, index=classes).round(3))

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for ax, (exp, o) in zip(axes.flat, outputs.items()):
    ConfusionMatrixDisplay.from_predictions(o["y_true"], o["y_pred"], labels=range(len(classes)),
                                            display_labels=classes, xticks_rotation=45, colorbar=False, ax=ax)
    ax.set_title(f"{exp}, seed 42")
plt.tight_layout()
plt.show()

## 7. Error analysis

The model with the best mean validation macro-F1 is chosen (never by test score). Its wrong test predictions are
grouped by confusion pair, and a few examples of the most frequent pairs are shown.

In [ ]:
best = results.groupby("model")["val_macro_f1"].mean().idxmax()
o = outputs[best]
errors = pd.DataFrame({"path": o["paths"], "true": [classes[i] for i in o["y_true"]],
                       "predicted": [classes[i] for i in o["y_pred"]], "confidence": o["confidence"]})
errors = errors[errors["true"] != errors["predicted"]]
pairs = errors.groupby(["true", "predicted"]).size().sort_values(ascending=False)
print(best, len(errors), "wrong test predictions")
print(pairs.head(10))

examples = pd.concat([errors[(errors["true"] == t) & (errors["predicted"] == p)].head(4) for t, p in pairs.index[:6]])
fig, axes = plt.subplots(-(-len(examples) // 6), 6, figsize=(16, 3 * -(-len(examples) // 6)), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, r in zip(axes.flat, examples.itertuples()):
    ax.imshow(Image.open(DATA_DIR / r.true / Path(r.path).name))
    ax.set_title(f"{r.true} -> {r.predicted}\n{r.confidence:.2f}", fontsize=8)
plt.tight_layout()
plt.show()